# Notebook 21 — Accuracy-first feature engineering

Development-only. Compare causal rolling volatility, downside risk, skew, drawdown and volume features using paired chronological folds. **Do not select confidence thresholds from outer test folds.**

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=="notebooks":ROOT=ROOT.parent
if not (ROOT/"src").exists():raise FileNotFoundError("Run from repository root or notebooks directory")
sys.path.insert(0,str(ROOT/"src"))
import pandas as pd
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.accuracy_benchmark import run_accuracy_benchmark

In [ ]:
SMOKE_TEST=True
N_FOLDS=2 if SMOKE_TEST else 5
MAX_TRAIN_ROWS=25000 if SMOKE_TEST else 120000
TREE_TRAIN_ROWS=15000 if SMOKE_TEST else 35000
OUTPUT=ROOT/"artifacts"/"accuracy_first"/"1d"
required=("master_labeled.parquet","latest_inference.parquet","transformers.joblib","manifest.json")
locations=[ROOT/"artifacts"/"master_tables"/"1d",ROOT/"notebooks"/"artifacts"/"master_tables"/"1d"]
source=next((p for p in locations if all((p/f).exists() for f in required)),None)
if source is None:raise FileNotFoundError("Notebook 03 artifacts missing")
master,latest,bundle,manifest=load_master(source)
print("Loaded",len(master),"rows",master.symbol.nunique(),"symbols")

## Paired chronological benchmark

Training quantiles, candle clustering, and scalers are fitted within each fold. The last 20% of timestamps are excluded before any feature comparison.

In [ ]:
scores,summary,selective,boundary=run_accuracy_benchmark(master,n_folds=N_FOLDS,max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,output_dir=OUTPUT)
print("Development cutoff:",boundary)
display(summary)

## Confidence versus coverage

These thresholds are diagnostics, **not** deployment decisions. Choosing a threshold based on these evaluation outcomes would introduce selection bias.

In [ ]:
display(selective.groupby(["scheme","model","threshold"],as_index=False).agg(coverage=("coverage","mean"),selective_accuracy=("selective_accuracy","mean"),n_selected=("n_selected","sum")).sort_values(["scheme","model","threshold"]))

## Next experiment

After reviewing fold consistency, select a small number of candidate feature families using **inner** chronological validation. Then evaluate selective directional accuracy and net-of-cost returns on a separate later period. Do not reopen the historical final holdout or overwrite prospective predictions.